In [7]:
from datetime import datetime, timedelta
import polars as pl
import numpy as np

from src.regimehmm import Regime, WalkForwardRegimeHMM
from src.data import YahooDataProvider

from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [ ]:
end = datetime.now()
start = end - timedelta(days=365 * 10)  # Last 10 years

tickers = {
    "^VIX": "vix",
    "^VIX3M": "vix3m",
    "^VVIX": "vvix",
    "TIP": "tip",
    "IEF": "ief",
    "SPY": "spy",
}

df = YahooDataProvider().get_historical_bars(
    symbols=list(tickers.keys()),
    start=start,
    end=end,
)

df = (df.with_columns(pl.col("symbol").replace(tickers))
        .unpivot(
            index=["date", "symbol"],
            variable_name="metric",
            value_name="value"
        )
        .with_columns(
            pl.concat_str(
                [
                    pl.col("symbol"),
                    pl.col("metric").str.to_lowercase()
                ],
                separator="_"
            ).alias("col_name")
        )
        .select(["date", "col_name", "value"])
        .pivot(values="value", index="date", on="col_name", sort_columns=True)
        .sort("date")
        .with_columns(pl.all().forward_fill()))

In [9]:
low_conf_threshold = 0.55

walker = WalkForwardRegimeHMM(
    training_window          = "1y",
    walk_forward_window      = "1mo",
    n_init                   = 10,
    n_iter                   = 1_000,
    random_state             = 42,
    min_holding_period       = 5,
    transition_penalty       = 0.25,
    low_confidence_threshold = low_conf_threshold,
)

annotated, wf_meta = walker.run(df, smoothed=False)
annotated = annotated.with_columns(
    block_id = (pl.col("regime") != pl.col("regime").shift()).fill_null(True).cum_sum()
)

wf_meta.tail()

C:\Users\ASUS\Projects\iv-band-strat\src\regimehmm.py:979: UserWarning: Training sample may be too small for the current feature set (n_obs=6, approx_free_params=111). Regime estimates can be unstable; consider a longer training window.
  model = model.fit(train_df)
C:\Users\ASUS\Projects\iv-band-strat\src\regimehmm.py:979: UserWarning: Training sample may be too small for the current feature set (n_obs=27, approx_free_params=111). Regime estimates can be unstable; consider a longer training window.
  model = model.fit(train_df)
C:\Users\ASUS\Projects\iv-band-strat\src\regimehmm.py:979: UserWarning: Training sample may be too small for the current feature set (n_obs=48, approx_free_params=111). Regime estimates can be unstable; consider a longer training window.
  model = model.fit(train_df)
C:\Users\ASUS\Projects\iv-band-strat\src\regimehmm.py:979: UserWarning: Training sample may be too small for the current feature set (n_obs=69, approx_free_params=111). Regime estimates can be unst

retrain_id,train_start,train_end,test_start,test_end,n_train,n_test,n_unknown,training_window,walk_forward_window
i64,datetime[μs],datetime[μs],datetime[μs],datetime[μs],i64,i64,i64,str,str
116,2024-11-04 00:00:00,2025-10-31 00:00:00,2025-11-03 00:00:00,2025-11-28 00:00:00,251,19,0,"""1y""","""1mo"""
117,2024-12-02 00:00:00,2025-11-28 00:00:00,2025-12-01 00:00:00,2025-12-31 00:00:00,251,22,0,"""1y""","""1mo"""
118,2025-01-02 00:00:00,2025-12-31 00:00:00,2026-01-02 00:00:00,2026-01-30 00:00:00,252,20,0,"""1y""","""1mo"""
119,2025-02-03 00:00:00,2026-01-30 00:00:00,2026-02-02 00:00:00,2026-02-27 00:00:00,251,19,0,"""1y""","""1mo"""
120,2025-03-03 00:00:00,2026-02-27 00:00:00,2026-03-02 00:00:00,2026-03-26 00:00:00,251,19,0,"""1y""","""1mo"""


In [10]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.02)

fig.add_trace(
    go.Scatter(
        x=annotated["date"].to_numpy(),
        y=annotated["spy_close"].log().to_numpy(),
        mode="lines",
        name="SPY",
        showlegend=False
    ), row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=annotated["date"].to_numpy(),
        y=annotated["vix_close"].to_numpy(),
        mode="lines",
        name="VIX",
        showlegend=False
    ), row=2, col=1
)

# Add shaded regions for regimes

regime_colors = {
    Regime.CALM: "green",
    Regime.CRISIS: "red",
    Regime.RECOVERY: "orange",
    Regime.STRESS: "purple",
}

regime_blocks = (
    annotated
    .group_by("block_id", "regime")
    .agg(
        start=pl.col("date").min(),
        end=pl.col("date").max()
    )
)

for block in regime_blocks.to_dicts():
    fig.add_vrect(
        x0=block["start"],
        x1=block["end"],
        fillcolor=regime_colors.get(block["regime"]),
        opacity=0.3,
        layer="below",
        line_width=0,
    )

# Add legend for regimes

for regime, color in regime_colors.items():
    fig.add_trace(
        go.Scatter(
            x=[None],
            y=[None],
            mode="markers",
            marker=dict(size=10, color=color),
            name=regime.name,
        )
    )
        
# Add confidence and margin plots
    
confidence = annotated["regime_confidence"].to_numpy()
margin = annotated["regime_margin"].to_numpy()
low_conf = confidence < low_conf_threshold
low_conf_y = np.where(low_conf, confidence, np.nan)


fig.add_trace(
    go.Scatter(
        x=annotated["date"].to_numpy(),
        y=confidence,
        mode="lines",
        name="Confidence (1 - normalized entropy)",
        line=dict(width=1.8),
    ),
    row=3,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=annotated["date"].to_numpy(),
        y=margin,
        mode="lines",
        name="Posterior margin (top1-top2)",
        line=dict(width=1.4, dash="dot"),
    ),
    row=3,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=annotated["date"].to_numpy(),
        y=low_conf_y,
        mode="markers",
        name="Low-confidence bars",
        marker=dict(size=5, opacity=0.6),
    ),
    row=3,
    col=1,
)

fig.add_hline(y=low_conf_threshold, line_dash="dash", row=3, col=1)

fig.update_layout(margin=dict(l=20, r=20, t=20, b=20))
fig.show()

In [11]:
import numpy as np
import plotly.figure_factory as ff

valid_labels = (
    annotated
    .filter(pl.col("regime") >= 0)
    .get_column("regime")
    .to_numpy()
    .astype(int)
)

trans_mat = np.zeros((len(Regime), len(Regime)), dtype=float)
for t in range(len(valid_labels) - 1):
    trans_mat[valid_labels[t], valid_labels[t + 1]] += 1.0

row_sums = trans_mat.sum(axis=1, keepdims=True)
row_sums[row_sums == 0.0] = 1.0
trans_prob = trans_mat / row_sums

regime_names = [r.name for r in Regime]
fig = ff.create_annotated_heatmap(
    z=trans_prob,
    x=regime_names,
    y=regime_names,
    annotation_text=np.vectorize(lambda x: f"{x:.1%}")(trans_prob),
    colorscale="YlOrRd",
    showscale=True,
)
fig.update_layout(
    xaxis_title="To",
    yaxis_title="From",
    margin=dict(l=50, r=30, t=60, b=40),
)
fig.show()

In [12]:
duration_df = (
    annotated
    .filter(pl.col("regime") >= 0)
    .with_columns(
        block_id=(pl.col("regime") != pl.col("regime").shift()).fill_null(True).cum_sum()
    )
    .group_by("block_id", "regime", "regime_name")
    .agg(
        start=pl.col("date").min(),
        end=pl.col("date").max(),
        duration_days=pl.len(),
    )
)

fig = go.Figure()
for regime in Regime:
    s = duration_df.filter(pl.col("regime") == regime.value)
    fig.add_trace(
        go.Box(
            y=s["duration_days"].to_numpy(),
            name=regime.name,
            marker_color=regime_colors[regime],
            boxmean=True,
        )
    )

fig.update_layout(
    title="Regime Duration Distribution",
    yaxis_title="Spell length (bars)",
    margin=dict(l=50, r=20, t=60, b=40),
)
fig.show()

(
    duration_df.group_by("regime_name")
    .agg(
        pl.col("duration_days").mean().round(2).alias("avg_bars"),
        pl.col("duration_days").median().alias("median_bars"),
        pl.col("duration_days").max().alias("max_bars"),
        pl.len().alias("n_spells"),
    )
    .sort("avg_bars", descending=True)
)

regime_name,avg_bars,median_bars,max_bars,n_spells
str,f64,f64,u32,u32
"""CALM""",23.62,19.5,82,32
"""RECOVERY""",23.28,19.0,85,29
"""STRESS""",16.27,10.0,95,33
"""CRISIS""",15.44,10.0,67,27
